In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import zarr
import os

path = r"/Volumes/data/Sasaki/backup_git/MTCargoSim/data/MTC/P0.5_A0.5/seed*.zarr"

polar_path = os.path.join(path, 'polar.zarr')
counts_path = os.path.join(path, 'counts.zarr')

polar_zarr = zarr.open(polar_path, mode = 'r')
counts_zarr = zarr.open(counts_path, mode = 'r')

polar_orders = polar_zarr[:]
counts = counts_zarr[:]

# 時間平均の表示
# 粒子がいるときだけの平均をとる（0を除外）
valid_indices = counts > 0
if np.any(valid_indices):
    avg_p = np.mean(polar_orders[valid_indices])
    print(f"📊 Average Local Polar Order: {avg_p:.3f}")
else:
    print("⚠️ No interaction detected throughout the simulation.")

# --- プロット ---
fig, ax1 = plt.subplots(figsize=(10, 6))

time_steps = np.arange(len(polar_orders))

# 左軸: ポーラー度
color = 'tab:red'
ax1.set_xlabel('Time Step (saved)')
ax1.set_ylabel('Local Polar Order $P$', color=color)
ax1.plot(time_steps, polar_orders, color=color, alpha=0.6, linewidth=1, label='Polar Order')
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_ylim(-0.05, 1.05)

# 移動平均（太線）
window = 100
if len(polar_orders) > window:
    ma = np.convolve(polar_orders, np.ones(window)/window, mode='valid')
    ax1.plot(np.arange(len(ma)) + window//2, ma, color='darkred', linewidth=2, label='Moving Avg')

# 右軸: 粒子数（参考用）
ax2 = ax1.twinx()  
color = 'tab:blue'
ax2.set_ylabel('Number of Neighbors', color=color)  
ax2.plot(time_steps, counts, color=color, alpha=0.15, linewidth=0.5, label='Count')
ax2.tick_params(axis='y', labelcolor=color)

plt.title(f'Local Polar Order around Cargo (r < $r_a$)')
fig.tight_layout()  

output_png = os.path.join(path, 'polar_counts.png')
output_pdf = os.path.join(path, 'polar_counts.pdf')

plt.savefig(output_png)
plt.savefig(output_pdf)

print(f"✅ Plot saved to: {output_png}")